# Prepare the Policy Summary Inputs

| Item | Description |
|---|---|
| Goal | Fix everything the policy-summary model run (T3.5.5) needs before any model call: the prompt wording, the per-year statement blocks for the map step, the reduce step that combines the per-year summaries, the token budget check, and the generation settings. |
| Input | `output/T3.5.2/topic68_statements.csv`, one row per Topic 68 statement in hearing order. |
| Output | `output/T3.5.4/policy_summary_prompt.md` (the prompts and settings, readable) and `output/T3.5.4/policy_summary_input.json` (the same plus the per-year statement blocks, read by T3.5.5). |
| Unit | Map step: one hearing year, all of its statements in one block, support and concern together. Reduce step: the per-year summaries in one block. |
| Model | `Qwen/Qwen3.5-4B` from the project's local snapshot, 262,144-token context. Generation as in the project's hearing summarizer: `do_sample=True`, `temperature=0.1`, reasoning traces off, output capped at 512 new tokens per call. |
| Length rule | The final summary must embed whole in `bge-large-en-v1.5`, so it must be at most 512 bge tokens. The reduce prompt asks for 200 to 250 words, which lands near 390 tokens and leaves headroom; T3.5.5 checks the count and re-asks for a shorter summary if needed. Each per-year summary is asked to be at most 150 words, shorter when the year has little to say. |
| Variables | • `statement_lines`: the statements as numbered lines `[k] (year, stance) text`, in hearing order.<br>• `chunks`: one per hearing year with statements, holding the statement numbers of that year and the model-token count of its rendered map prompt. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how the summary is used downstream.


## Step 1. Import packages

Read the statement table, count model tokens with the model's own tokenizer, and write the input files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer
from IPython.display import Markdown, display


## Step 2. Define file locations and settings

All numbers the model run depends on are named here and copied into the input file, so the run never retypes them.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.5.4"  # the identifier names this notebook's output folder
STATEMENTS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.5.2/topic68_statements.csv"
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--Qwen--Qwen3.5-4B"
    "/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
)
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

TOPIC_ID = 68
TOPIC_LABEL = "consumer data privacy legislation"
INPUT_TOKEN_BUDGET = 32_000  # model tokens per call; far below the 262,144 context, keeps every call fast on one GPU
MAP_TARGET_WORDS = "at most 150"  # per-year summary; a year with one or two statements should come out much shorter
REDUCE_TARGET_WORDS = "200 to 250"  # a range lands inside the 512-token limit with headroom (about 1.55 tokens per word)
MAX_NEW_TOKENS = 512  # hard cap on generated tokens per call
MAX_BGE_TOKENS = 512  # the encoder's input limit the final summary must respect
GENERATION = {"do_sample": True, "temperature": 0.1}  # as in the project's hearing summarizer
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.4


## Step 3. Load the statements

Keep hearing order so each year's block reads in the order the hearings were held.


In [3]:
statements = pd.read_csv(STATEMENTS_FILE)
assert statements["text"].notna().all()
assert statements["stance"].isin(["support", "concern"]).all()
statements = statements.sort_values(["hearing_year", "packageId", "turn_idx", "record_idx"]).reset_index(drop=True)
year_min, year_max = int(statements["hearing_year"].min()), int(statements["hearing_year"].max())
print(f"Statements: {len(statements):,}; hearings {year_min}–{year_max}; by stance: {statements['stance'].value_counts().to_dict()}")
print(f"Years with statements: {statements['hearing_year'].nunique()}")
display(statements[["hearing_year", "stance", "n_tokens"]].head(3))


Statements: 374; hearings 1999–2025; by stance: {'support': 221, 'concern': 153}
Years with statements: 26


,hearing_year,stance,n_tokens
0,1999,support,39
1,1999,support,48
2,1999,support,48


## Step 4. Format the statement lines

One line per statement: its number, the hearing year, the stance, and the text. The number lets the model refer to statements without quoting them, and the stance is given so the model does not infer it.


In [4]:
statement_lines = [
    f"[{number}] ({row.hearing_year}, {row.stance}) {row.text}"
    for number, row in enumerate(statements.itertuples(index=False), start=1)
]
statements["number"] = range(1, len(statements) + 1)
statements["line"] = statement_lines
print(f"Lines: {len(statement_lines):,}")
print("\n".join(statement_lines[:3]))


Lines: 374
[1] (1999, support) The bill is an important first step that recognizes that as the Internet becomes more of a presence in global commerce there have to be guarantees and assurances that business and personal information remains confidential.
[2] (1999, support) With all due respect, this process is too long, which is why in the E-PRIVACY bill we give the administration a one-time 15-day review of products that are generally available before they are permitted to export them.
[3] (1999, support) The second area where I think the PROTECT Act can go farther is the explicit delineation of the rights and procedural protections of Americans in their ability to use encryption and to be secure in their use of encrypted data.


## Step 5. Write the prompts

The system prompt fixes the role. The map prompt summarizes one year's statements and is told to write less when the year says little, so a year with one statement does not get padded. The reduce prompt combines the per-year summaries into the policy summary: what the legislation asks of organizations and what it tries to regulate, in continuous prose, with the two failure modes named, a summary that only says the topic is about laws, and knowledge brought in from outside the statements.


In [5]:
SYSTEM_PROMPT = (
    "You are a policy analyst with a working knowledge of United States congressional hearings. "
    "You report only what the statements say."
)

MAP_PROMPT_TEMPLATE = """Below are {n_statements} statements that members of the House Science and Senate Commerce committees made in hearings held in {year} on the topic "{topic_label}". Each statement is numbered and marked with the legislator's stance toward the topic, support or concern.

--- STATEMENTS ---
{statements_block}
--- END OF STATEMENTS ---

Summarize, in {target_words} words, what the legislation discussed in these statements asks of companies and other organizations and what it tries to regulate: which data practices, which kinds of data, and which actors. Write continuous prose. If the statements say little, write only a sentence or two. Do not list or count the statements, do not name legislators, and do not add knowledge beyond the statements."""

REDUCE_PROMPT_TEMPLATE = """Below are {n_years} summaries, one per year, of statements that members of the House Science and Senate Commerce committees made in hearings held between {year_min} and {year_max} on the topic "{topic_label}". Each summary is marked with its year.

--- YEARLY SUMMARIES ---
{summaries_block}
--- END OF YEARLY SUMMARIES ---

Combine them into one summary of the legislation, in {target_words} words, for a reader who has not seen the statements. Cover two things:
1. What the legislation asks of companies and other organizations: the obligations it imposes on them and the rights it gives consumers.
2. What it tries to regulate: which data practices, which kinds of data, and which actors.

Write continuous prose in one or two paragraphs. Name the specific requirements, data practices, and actors; do not say that the topic is about laws, bills, or privacy in general. Do not add knowledge beyond the yearly summaries."""

display(Markdown(
    "**System prompt**\n\n" + SYSTEM_PROMPT
    + "\n\n**Map prompt template, one call per year**\n\n```\n" + MAP_PROMPT_TEMPLATE
    + "\n```\n\n**Reduce prompt template, one call**\n\n```\n" + REDUCE_PROMPT_TEMPLATE + "\n```"
))


**System prompt**

You are a policy analyst with a working knowledge of United States congressional hearings. You report only what the statements say.

**Map prompt template, one call per year**

```
Below are {n_statements} statements that members of the House Science and Senate Commerce committees made in hearings held in {year} on the topic "{topic_label}". Each statement is numbered and marked with the legislator's stance toward the topic, support or concern.

--- STATEMENTS ---
{statements_block}
--- END OF STATEMENTS ---

Summarize, in {target_words} words, what the legislation discussed in these statements asks of companies and other organizations and what it tries to regulate: which data practices, which kinds of data, and which actors. Write continuous prose. If the statements say little, write only a sentence or two. Do not list or count the statements, do not name legislators, and do not add knowledge beyond the statements.
```

**Reduce prompt template, one call**

```
Below are {n_years} summaries, one per year, of statements that members of the House Science and Senate Commerce committees made in hearings held between {year_min} and {year_max} on the topic "{topic_label}". Each summary is marked with its year.

--- YEARLY SUMMARIES ---
{summaries_block}
--- END OF YEARLY SUMMARIES ---

Combine them into one summary of the legislation, in {target_words} words, for a reader who has not seen the statements. Cover two things:
1. What the legislation asks of companies and other organizations: the obligations it imposes on them and the rights it gives consumers.
2. What it tries to regulate: which data practices, which kinds of data, and which actors.

Write continuous prose in one or two paragraphs. Name the specific requirements, data practices, and actors; do not say that the topic is about laws, bills, or privacy in general. Do not add knowledge beyond the yearly summaries.
```

## Step 6. Build one chunk per hearing year and check the budget

Render each year's map prompt exactly as the model run will, with the chat template and reasoning traces off, and count its tokens with the model's tokenizer. Every year must fit the per-call budget.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)

chunks = []
for hearing_year, year_statements in statements.groupby("hearing_year", sort=True):
    statements_block = "\n".join(year_statements["line"])
    user_prompt = MAP_PROMPT_TEMPLATE.format(
        n_statements=len(year_statements), year=hearing_year, topic_label=TOPIC_LABEL,
        statements_block=statements_block, target_words=MAP_TARGET_WORDS,
    )
    rendered_prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    chunks.append({
        "chunk_id": len(chunks) + 1,
        "hearing_year": int(hearing_year),
        "n_statements": int(len(year_statements)),
        "statement_numbers": year_statements["number"].tolist(),
        "n_model_tokens": len(tokenizer(rendered_prompt)["input_ids"]),
    })
chunk_table = pd.DataFrame(chunks).drop(columns="statement_numbers")
assert chunk_table["n_statements"].sum() == len(statements)
assert chunk_table["n_model_tokens"].le(INPUT_TOKEN_BUDGET).all()
print(f"Map calls: {len(chunks)}, one per year with statements; largest prompt {chunk_table['n_model_tokens'].max():,} model tokens against a budget of {INPUT_TOKEN_BUDGET:,}")
display(chunk_table.set_index("chunk_id").T)


Map calls: 26, one per year with statements; largest prompt 3,512 model tokens against a budget of 32,000


chunk_id,1,2,3,4,5,6,7,8,9,10,...,17,18,19,20,21,22,23,24,25,26
hearing_year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
n_statements,9,33,15,13,8,4,2,8,1,6,...,21,11,44,40,6,62,1,10,9,6
n_model_tokens,683,1666,1100,1350,648,547,346,606,281,612,...,1496,911,2463,2392,503,3512,235,806,884,508


## Step 7. Save the prompt file and the input file

The Markdown file is for reading; the JSON file is what T3.5.5 reads. Both carry the same prompts and settings.


In [7]:
prompt_file = OUTPUT_FOLDER / "policy_summary_prompt.md"
input_file = OUTPUT_FOLDER / "policy_summary_input.json"

settings = {
    "model_folder": str(MODEL_FOLDER),
    "generation": GENERATION,
    "enable_thinking": False,
    "max_new_tokens": MAX_NEW_TOKENS,
    "map_target_words": MAP_TARGET_WORDS,
    "reduce_target_words": REDUCE_TARGET_WORDS,
    "max_bge_tokens": MAX_BGE_TOKENS,
    "input_token_budget": INPUT_TOKEN_BUDGET,
}
prompt_file.write_text(
    "# Policy summary prompts and settings\n\n"
    f"Topic {TOPIC_ID}, {TOPIC_LABEL}; {len(statements)} statements, hearings {year_min}–{year_max}; "
    f"map by year ({len(chunks)} calls), then one reduce call.\n\n"
    "## Settings\n\n```json\n" + json.dumps(settings, indent=2) + "\n```\n\n"
    "## System prompt\n\n" + SYSTEM_PROMPT + "\n\n"
    "## Map prompt template, one call per year\n\n```\n" + MAP_PROMPT_TEMPLATE + "\n```\n\n"
    "## Reduce prompt template, one call\n\n```\n" + REDUCE_PROMPT_TEMPLATE + "\n```\n",
    encoding="utf-8",
)
policy_summary_input = {
    "topic_id": TOPIC_ID,
    "topic_label": TOPIC_LABEL,
    "n_statements": len(statements),
    "year_min": year_min,
    "year_max": year_max,
    "mode": "map_by_year_then_reduce",
    "chunks": chunks,
    "statement_lines": statement_lines,
    "prompts": {
        "system": SYSTEM_PROMPT,
        "map_template": MAP_PROMPT_TEMPLATE,
        "reduce_template": REDUCE_PROMPT_TEMPLATE,
    },
    "settings": settings,
}
input_file.write_text(json.dumps(policy_summary_input, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {prompt_file}")
print(f"Saved: {input_file}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.4/policy_summary_prompt.md
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.5.4/policy_summary_input.json
